In [ ]:
import numpy as np
import pandas as pd

# Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

image_size = 64
batch_size = 64
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

val_ratio = 0.2 # train 80%, val 20%

# 전체 길이 기준으로 train/val 길이 계산
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SEModule(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class ResNeXtBasicBlock(nn.Module):
    """
    ResNet34 스타일의 BasicBlock + ResNeXt-style grouped conv.
    - conv1, conv2 둘 다 3x3 group conv
    - expansion = 1 (ResNet34와 동일)
    """
    expansion = 1

    def __init__(self,
                 inplanes,
                 planes,
                 stride=1,
                 downsample=None,
                 cardinality=8,
                 base_width=64,
                 reduction=16):
        super().__init__()

        # ResNeXt width 계산
        # D = planes * (base_width / 64), 그 뒤 그룹 수(cardinality)만큼 채널 확장
        D = int(planes * (base_width / 64.0))
        C = cardinality

        # 첫 번째 3x3 group conv
        self.conv1 = nn.Conv2d(
            inplanes, D * C,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=C,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(D * C)

        # 두 번째 3x3 group conv
        # 출력 채널은 planes (expansion=1)
        self.conv2 = nn.Conv2d(
            D * C, planes * self.expansion,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=C,
            bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes * self.expansion)

        self.se = SEModule(planes, reduction)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

        self.cardinality = cardinality
        self.base_width = base_width

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = self.se(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

class ResNeXt34(nn.Module):
    """
    ResNet34와 같은 깊이(3,4,6,3 blocks)를 갖는 ResNeXt.
    64x64 / CIFAR 스타일 입력을 가정:
      - stem: 3x3 conv, stride 1, no maxpool
      - layer1~4: 첫 블록에서만 stride=2 (layer2~4)
    """
    def __init__(self,
                 num_classes=15,
                 cardinality=8,     # 그룹 수 (예: 8, 16 등)
                 base_width=64):    # 그룹별 width 조절
        super().__init__()
        self.inplanes = 64
        self.cardinality = cardinality
        self.base_width = base_width

        # stem: 64x64 -> 64x64
        self.conv1 = nn.Conv2d(
            3, 64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        # CIFAR/64x64에서는 보통 maxpool 생략
        # self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet34와 같은 block 수: [3, 4, 6, 3]
        self.layer1 = self._make_layer(64,  3, stride=1)  # 64x64 유지
        self.layer2 = self._make_layer(128, 4, stride=2)  # 32x32
        self.layer3 = self._make_layer(256, 6, stride=2)  # 16x16
        self.layer4 = self._make_layer(512, 3, stride=2)  # 8x8

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * ResNeXtBasicBlock.expansion, num_classes)

        # 초기화 (간단한 Kaiming)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_out', nonlinearity='relu'
                )
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, blocks, stride=1):
        """
        planes : 이 stage의 output 채널 수 (64,128,256,512)
        blocks : 해당 stage의 block 개수 (3,4,6,3)
        stride : 첫 block의 stride (downsample용)
        """
        downsample = None
        if stride != 1 or self.inplanes != planes * ResNeXtBasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.inplanes,
                    planes * ResNeXtBasicBlock.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(planes * ResNeXtBasicBlock.expansion),
            )

        layers = []
        # 첫 block: stride 적용 + 채널 수 변경 + downsample
        layers.append(
            ResNeXtBasicBlock(
                self.inplanes,
                planes,
                stride=stride,
                downsample=downsample,
                cardinality=self.cardinality,
                base_width=self.base_width,
            )
        )
        self.inplanes = planes * ResNeXtBasicBlock.expansion

        # 나머지 blocks: stride=1, downsample 없음
        for _ in range(1, blocks):
            layers.append(
                ResNeXtBasicBlock(
                    self.inplanes,
                    planes,
                    stride=1,
                    downsample=None,
                    cardinality=self.cardinality,
                    base_width=self.base_width,
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        # x: (N, 3, 64, 64)
        x = self.conv1(x)   # (N, 64, 64, 64)
        x = self.bn1(x)
        x = self.relu(x)
        # x = self.maxpool(x)  # 쓰고 싶으면 활성화

        x = self.layer1(x)  # 64x64
        x = self.layer2(x)  # 32x32
        x = self.layer3(x)  # 16x16
        x = self.layer4(x)  #  8x8

        x = self.avgpool(x)  # (N, 512, 1, 1)
        x = torch.flatten(x, 1)  # (N, 512)
        x = self.fc(x)           # (N, num_classes)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 15
model = ResNeXt34(num_classes=num_classes, cardinality=32, base_width=64).to(device)

# Model parameter checking

In [ ]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

# Model training

In [ ]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

In [9]:
import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast

# 0. 벤치마크 켜기 (속도 향상)
torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler('cuda')

epochs = 15
save_path = "best_model.pth"
best_val_loss = float("inf")

criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # 통계 (AMP로 계산된 output을 그대로 사용)
        train_loss_sum += loss.item() * y.size(0)
        
        # 예측값 계산 (여기는 그라디언트 필요 없으므로 detach 추천)
        preds = torch.argmax(output.detach(), dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # VALIDATION (그대로 유지)
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # 검증 때는 AMP를 굳이 안 써도 되지만, 쓰면 조금 더 빠를 수 있음
            with autocast():
                output = model(x)
                val_loss_batch = criterion(output, y)

            val_loss_sum += val_loss_batch.item() * y.size(0)
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    scheduler.step()

    # MODEL SAVE
    save_dict = {
        "epoch": epoch,
        "model_state_dict": (
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict()
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    torch.save(save_dict, f'epoch_{epoch}.pth')

    # BEST MODEL SPECIFICATION
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        save_dict = {
            "epoch": epoch,
            "model_state_dict": (
                model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict()
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
        }

        torch.save(save_dict, save_path)
        print(f"Best model saved at epoch {epoch} (val_loss={val_loss:.4f})")

Epoch 0 [Train]:   1%|          | 6/562 [01:52<2:53:26, 18.72s/it]


KeyboardInterrupt: 

In [ ]:
'''
# 모델 불러오기
checkpoint = torch.load("best_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = ConvNeXtBN(num_classes=15)
base_model.load_state_dict(checkpoint["model_state_dict"])

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
'''

In [ ]:
'''
# 모델 로드 검증
missing_keys, unexpected_keys = base_model.load_state_dict(
    checkpoint["model_state_dict"], strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# 파라미터 값 확인
with torch.no_grad():
    w = base_model.downsample_layers[0][0].weight

print("Sample weight stats:")
print("  mean:", w.mean().item())
print("  std :", w.std().item())
print("  min :", w.min().item())
print("  max :", w.max().item())

# forward 테스트
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)

print("Output shape:", out.shape)
print("Has NaN:", torch.isnan(out).any().item())
'''

In [ ]:
'''
print("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))
print("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))
print("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))
'''

# Submit
Do not edit the submission code below.

In [ ]:
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)